In [17]:
pd.set_option('future.no_silent_downcasting', True)

In [18]:
with open(r'B:\PBCS\VI Sem\Machine Learning\MiniProject\smartflow-energy-optimizer\data\raw\simulated_sensor_data.csv', 'w', encoding='utf-8') as f:
    f.write('timestamp,room_id,occupancy,fan_status,light_status,ac_status,power_consumption_w,day_of_week,is_break_period,is_after_hours,wastage_label\n')
print('Done! File recreated with correct encoding.')

Done! File recreated with correct encoding.


In [19]:
import pandas as pd
import numpy as np

# Load both datasets
manual = pd.read_csv('../data/raw/manual_observations.csv')
simulated = pd.read_csv('../data/raw/simulated_sensor_data.csv')

# Check what we have
print("Manual rows:", len(manual))
print("Simulated rows:", len(simulated))
print(manual.head())
print(simulated.head())

Manual rows: 250
Simulated rows: 0
         date time_slot room_id  occupancy  fan_status  light_status  \
0  02-02-2026   2:00 PM  Class1          1           1             1   
1  02-02-2026   3:00 PM    Lab1          1           1             1   
2  02-02-2026   4:30 PM    Lab2          0           1             1   
3  02-02-2026   4:30 PM  Class3          0           1             1   
4  02-02-2026   9:00 AM    Lab2          0           0             0   

   ac_status  projector_status  power_consumption_w day_of_week  \
0          0                 0                  460      Monday   
1          0                 0                  460      Monday   
2          0                 0                  460      Monday   
3          0                 0                  460      Monday   
4          0                 0                    0      Monday   

   is_break_period  is_after_hours  wastage_label  
0                0               0              0  
1                0       

In [20]:
# Check for missing values
print(manual.isnull().sum())
print(simulated.isnull().sum())

# Drop rows with missing values (simple approach)
manual = manual.dropna()
simulated = simulated.dropna()

# Merge both datasets
df = pd.concat([manual, simulated], ignore_index=True)
print("Total rows after merge:", len(df))

date                   0
time_slot              0
room_id                0
occupancy              0
fan_status             0
light_status           0
ac_status              0
projector_status       0
power_consumption_w    0
day_of_week            0
is_break_period        0
is_after_hours         0
wastage_label          0
dtype: int64
timestamp              0
room_id                0
occupancy              0
fan_status             0
light_status           0
ac_status              0
power_consumption_w    0
day_of_week            0
is_break_period        0
is_after_hours         0
wastage_label          0
dtype: int64
Total rows after merge: 250


In [21]:
# Combine date + time_slot into a proper timestamp
df['timestamp'] = pd.to_datetime(
    df['date'] + ' ' + df['time_slot'], 
    format='mixed'
)

# Now extract hour and day number
df['hour_of_day'] = df['timestamp'].dt.hour
df['day_of_week_num'] = df['timestamp'].dt.dayofweek  # 0=Monday

# is_break_period: 10:30 AM (hour=10) or 1:00 PM (hour=13)
df['is_break_period'] = (
    (df['hour_of_day'] == 10) | 
    (df['hour_of_day'] == 13)
).astype(int)

# is_after_hours: after 4 PM (your college ends at 4)
df['is_after_hours'] = (df['hour_of_day'] >= 16).astype(int)

# Sort by timestamp before rolling
df = df.sort_values('timestamp').reset_index(drop=True)

# Rolling average
df['rolling_avg_30min'] = df['power_consumption_w'].rolling(window=3, min_periods=1).mean()

# Lag feature
df['lag_1h_power'] = df['power_consumption_w'].shift(1).fillna(0)

# Verify - should show actual hour numbers now
print(df[['date','time_slot','hour_of_day','is_break_period','is_after_hours','rolling_avg_30min']].head(10))

         date time_slot  hour_of_day  is_break_period  is_after_hours  \
0  01-04-2026   8:30 AM            8                0               0   
1  01-04-2026  10:00 AM           10                1               0   
2  01-04-2026  10:45 AM           10                1               0   
3  02-02-2026   9:00 AM            9                0               0   
4  02-02-2026  12:30 PM           12                0               0   
5  02-02-2026   2:00 PM           14                0               0   
6  02-02-2026   3:00 PM           15                0               0   
7  02-02-2026   4:30 PM           16                0               1   
8  02-02-2026   4:30 PM           16                0               1   
9  02-03-2026   1:00 PM           13                1               0   

   rolling_avg_30min  
0         460.000000  
1         460.000000  
2         421.666667  
3         268.333333  
4         168.333333  
5         206.666667  
6         360.000000  
7         46

In [22]:
# Save merged dataset
df.to_csv('../data/processed/merged_dataset.csv', index=False)

# Select only the final features for ML training
features = ['hour_of_day', 'day_of_week_num', 'is_break_period',
            'is_after_hours', 'occupancy', 'fan_status', 'light_status',
            'power_consumption_w', 'rolling_avg_30min', 'lag_1h_power',
            'wastage_label']

df_final = df[features].dropna()
df_final.to_csv('../data/processed/features_final.csv', index=False)
print("Saved! Shape:", df_final.shape)

Saved! Shape: (250, 11)
